In [2]:
import torch
import open_clip
import pandas as pd
from collections import defaultdict
from basemodel import load_finetuned_timm_wrapper

In [ ]:
def get_vit_architecture(model, model_name="ViT"):
    """
    Parses a Vision Transformer model and returns a structured list of its layers.
    This version is more robust to handle naming differences in patch embedding and MLP layers
    between DINO (timm) and OpenCLIP.
    """
    arch_list = []

    # --- Find the core ViT parts ---
    if hasattr(model, "visual"):  # Handle full OpenCLIP model
        vit_model = model.visual
    else:
        vit_model = model

    # --- Identify key components by type and name ---
    patch_embed = vit_model.patch_embed if hasattr(vit_model, "patch_embed") else vit_model.conv1
    cls_token = vit_model.cls_token if hasattr(vit_model, "cls_token") else vit_model.class_embedding
    pos_embed = vit_model.pos_embed if hasattr(vit_model, "pos_embed") else vit_model.positional_embedding

    if hasattr(vit_model, "blocks"):
        blocks = vit_model.blocks
    elif hasattr(vit_model, "transformer"):
        blocks = vit_model.transformer.resblocks
    else:
        raise ValueError("Could not find transformer blocks.")

    # --- Build the structured list ---

    # CORRECTED: Handle patch embedding details
    if hasattr(patch_embed, "proj"):  # DINO style
        patch_embed_details = f"Output features: {patch_embed.proj.out_channels}"
    else:  # OpenCLIP style (it's a raw Conv2d)
        patch_embed_details = f"Output features: {patch_embed.out_channels}"

    arch_list.append(
        {
            "Component": "Input",
            "Layer Name": "patch_embed" if hasattr(vit_model, "patch_embed") else "conv1",
            "Layer Type": type(patch_embed).__name__,
            "Details": patch_embed_details,
        }
    )

    arch_list.append(
        {
            "Component": "Input",
            "Layer Name": "cls_token" if hasattr(vit_model, "cls_token") else "class_embedding",
            "Layer Type": "Parameter",
            "Details": f"Shape: {cls_token.shape}",
        }
    )

    arch_list.append(
        {
            "Component": "Input",
            "Layer Name": "pos_embed" if hasattr(vit_model, "pos_embed") else "positional_embedding",
            "Layer Type": "Parameter",
            "Details": f"Shape: {pos_embed.shape}",
        }
    )

    if hasattr(vit_model, "ln_pre"):
        arch_list.append(
            {
                "Component": "Pre-Blocks",
                "Layer Name": "ln_pre",
                "Layer Type": type(vit_model.ln_pre).__name__,
                "Details": "",
            }
        )

    for i, block in enumerate(blocks):
        block_prefix = f"Block {i}"

        norm1 = block.norm1 if hasattr(block, "norm1") else block.ln_1
        attn = block.attn
        norm2 = block.norm2 if hasattr(block, "norm2") else block.ln_2
        mlp = block.mlp

        arch_list.append(
            {
                "Component": block_prefix,
                "Layer Name": "norm1" if hasattr(block, "norm1") else "ln_1",
                "Layer Type": type(norm1).__name__,
                "Details": "",
            }
        )
        arch_list.append(
            {
                "Component": block_prefix,
                "Layer Name": "attn",
                "Layer Type": type(attn).__name__,
                "Details": f"Heads: {attn.num_heads}",
            }
        )

        if hasattr(block, "ls1"):
            arch_list.append(
                {
                    "Component": block_prefix,
                    "Layer Name": "ls1 (LayerScale)",
                    "Layer Type": type(block.ls1).__name__,
                    "Details": f"Dim: {block.ls1.gamma.shape[0]}",
                }
            )

        arch_list.append(
            {
                "Component": block_prefix,
                "Layer Name": "norm2" if hasattr(block, "norm2") else "ln_2",
                "Layer Type": type(norm2).__name__,
                "Details": "",
            }
        )

        # CORRECTED: Handle MLP details
        if hasattr(mlp, "fc2"):  # DINO style
            mlp_details = f"Out Features: {mlp.fc2.out_features}"
        elif hasattr(mlp, "c_proj"):  # OpenCLIP style
            mlp_details = f"Out Features: {mlp.c_proj.out_features}"
        else:
            mlp_details = "N/A"

        arch_list.append(
            {"Component": block_prefix, "Layer Name": "mlp", "Layer Type": type(mlp).__name__, "Details": mlp_details}
        )

        if hasattr(block, "ls2"):
            arch_list.append(
                {
                    "Component": block_prefix,
                    "Layer Name": "ls2 (LayerScale)",
                    "Layer Type": type(block.ls2).__name__,
                    "Details": f"Dim: {block.ls2.gamma.shape[0]}",
                }
            )

    final_norm = vit_model.norm if hasattr(vit_model, "norm") else vit_model.ln_post
    arch_list.append(
        {
            "Component": "Output",
            "Layer Name": "norm" if hasattr(vit_model, "norm") else "ln_post",
            "Layer Type": type(final_norm).__name__,
            "Details": "",
        }
    )

    if hasattr(vit_model, "proj"):
        arch_list.append(
            {
                "Component": "Output",
                "Layer Name": "proj",
                "Layer Type": "Parameter",
                "Details": f"Shape: {vit_model.proj.shape}",
            }
        )

    return pd.DataFrame(arch_list)

In [ ]:
CHECKPOINT_PATH = (
    "/workspaces/bachelor_thesis_code/giantbodybest74ens82.pth"
)
IMG_SIZE = 518
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE = "vit_giant_patch14_dinov2.lvd142m"
EMBEDDING_DIM = 256  # The output size you trained for


model_wrapper, weights = load_finetuned_timm_wrapper(
    checkpoint_path=CHECKPOINT_PATH,
    backbone_name=BACKBONE,
    embedding_size=EMBEDDING_DIM,
    image_size=IMG_SIZE,
    device=DEVICE,
)

dino_model = model_wrapper.model
dino_wrapper = model_wrapper

In [ ]:
modules = []
for name, module in dino_wrapper.named_modules():
    key = f"model.{name}" 
    modules.append(key)


for name, module in dino_wrapper.embedding_layer.named_modules():
    key = f"embedding_layer.{name}"
    modules.append(key)

for module in modules:
    print(f"module: {module}")

In [ ]:
print(dino_wrapper.embedding_layer)
print(type(dino_wrapper.embedding_layer))


In [ ]:
clip_model_full, _, _ = open_clip.create_model_and_transforms("ViT-g-14", pretrained="laion2b_s34b_b88k")
clip_model = clip_model_full.visual

In [12]:
dino_df = get_vit_architecture(dino_model, model_name="DINO")
openclip_df = get_vit_architecture(clip_model, model_name="OpenCLIP")

In [13]:
def create_canonical_name(row):
    name = row["Layer Name"].split(" ")[0]
    return f"{row['Component']}::{name}"

In [ ]:
dino_df["Canonical Name"] = dino_df.apply(create_canonical_name, axis=1)
openclip_df["Canonical Name"] = openclip_df.apply(create_canonical_name, axis=1)

# Merge the dataframes on the canonical name for a side-by-side view
comparison_df = pd.merge(dino_df, openclip_df, on="Canonical Name", how="outer", suffixes=("_DINO", "_OpenCLIP"))

# Tidy up the final dataframe for display
comparison_df = comparison_df.set_index("Canonical Name")
display_cols = ["Layer Name_DINO", "Layer Type_DINO", "Layer Name_OpenCLIP", "Layer Type_OpenCLIP"]
comparison_df = comparison_df[display_cols]

with pd.option_context("display.max_rows", None, "display.width", 120):
    print(comparison_df)

In [ ]:
dino_block = dino_model.blocks[0]

print("\n=== DINO Transformer Block[0] Structure Overview ===\n")

for name, module in dino_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()

In [ ]:
print("--- DINOv2 Image Backbone ---\n")

print("1) Patch Embedding (`model.patch_embed`)\n")
# Print the sub-modules of the patch_embed layer
for name, module in dino_wrapper.model.patch_embed.named_children():
    print(f"({name}) {module.__class__.__name__}:")
    print(module, "\n")


print("\n2) Transformer Blocks (`model.blocks`)\n")
print(f"The model has {len(dino_wrapper.model.blocks)} identical blocks. Showing Block[0] as a representative example:\n")

# Isolate the first block for detailed printing
dino_block = dino_wrapper.model.blocks[0]
print("--- DINO Transformer Block[0] Structure ---\n")
for name, module in dino_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()


print("\n3) Final Head (`model.head`)\n")
print("norm:", dino_wrapper.model.norm)
print("fc_norm:", dino_wrapper.model.fc_norm)
print("head:", dino_wrapper.model.head)
print("\n" + "=" * 60 + "\n")


print("--- Separate Feature Embedding Layer ---\n")
print(dino_wrapper.embedding_layer)
print("\n" + "=" * 60 + "\n")

In [ ]:
clip_block = clip_model.transformer.resblocks[0]

print("\n=== OPEN CLIP Transformer Block[0] Structure Overview ===\n")

for name, module in clip_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()

In [ ]:
from torchvision.models import vision_transformer

vit_model = vision_transformer.vit_b_16(pretrained=True)
vit_encoder_block = vit_model.encoder.layers[0]

print("\n=== Standart VIT Transformer Block[0] Structure Overview ===\n")

for name, module in vit_encoder_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()

In [ ]:
from transformers import BertModel

bert = BertModel.from_pretrained("bert-base-uncased")
encoder_block = bert.encoder.layer[0]

print("\n=== BERT Transformer Block[0] Structure Overview ===\n")

for name, module in encoder_block.named_children():
    print(f"--- {name} ---")
    print(module)
    print()

In [ ]:
print("\n=== DINO Model Overview ===\n")
print(dino_model)
print("\n=== OpenCLIP Model Overview ===\n")
print(clip_model_full)
print("\n=== Standart VIT Model Overview ===\n")
print(vit_model)